# Project 18 - Notebook 01: Data Preparation & Exploratory Data Analysis (EDA)
Khảo sát dữ liệu ExDark, kiểm tra phân bố 12 classes và trực quan hóa bounding boxes trong đêm.

In [ ]:
import os
import yaml
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

DATASET_DIR = Path('../Dataset/exdark_yolo_dark')
yaml_path = DATASET_DIR / 'data.yaml'

with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

classes = cfg['names']
print(f'Classes count: {len(classes)}')
print(classes)

### 1. Thống kê số lượng ảnh và nhãn theo từng split

In [ ]:
for split in ['train', 'valid', 'test']:
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    n_imgs = len(list(img_dir.glob('*.*')))
    n_lbls = len(list(lbl_dir.glob('*.txt')))
    print(f'{split.capitalize():5s}: {n_imgs} ảnh, {n_lbls} file nhãn')

### 2. Trực quan hóa ảnh tối mẫu kèm Bounding Box

In [ ]:
train_imgs = list((DATASET_DIR / 'train' / 'images').glob('*.*'))
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, ax in enumerate(axes):
    img_path = train_imgs[i]
    lbl_path = DATASET_DIR / 'train' / 'labels' / f'{img_path.stem}.txt'
    
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    if lbl_path.exists():
        with open(lbl_path, 'r') as f:
            for line in f:
                cls_id, xc, yc, bw, bh = map(float, line.strip().split()[:5])
                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)
                cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img_rgb, classes[int(cls_id)], (x1, max(y1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                
    ax.imshow(img_rgb)
    ax.set_title(f'Sample {i+1}: {img_path.name[:15]}...')
    ax.axis('off')

plt.tight_layout()
plt.show()